In [ ]:
import pandas as pd
import numpy as np
import statistics as stat
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [ ]:
df_customers = pd.read_csv(os.path.join('data','olist_customers_dataset.csv'))
df_customers

In [ ]:
#olist_geolocation_dataset, olist_order_items_dataset, olist_order_payments_dataset, olist_order_reviews_dataset,
#olist_orders_dataset, olist_products_dataset, olist_sellers_dataset, product_category_name_translation

In [ ]:
df_geoloc = pd.read_csv(os.path.join('data','olist_geolocation_dataset.csv'))
df_order_item = pd.read_csv(os.path.join('data','olist_order_items_dataset.csv'))
df_order_pay = pd.read_csv(os.path.join('data','olist_order_payments_dataset.csv'))
df_order_rew = pd.read_csv(os.path.join('data','olist_order_reviews_dataset.csv'))
df_orders = pd.read_csv(os.path.join('data','olist_orders_dataset.csv'))
df_prod = pd.read_csv(os.path.join('data','olist_products_dataset.csv'))
df_sell = pd.read_csv(os.path.join('data','olist_sellers_dataset.csv'))
df_prod_cat_name_tr = pd.read_csv(os.path.join('data','product_category_name_translation.csv'))

In [ ]:
print(df_customers.columns.tolist())

In [ ]:
df_geoloc

In [ ]:
df_order_item

In [ ]:
df_order_pay

In [ ]:
df_order_rew

In [ ]:
df_orders

In [ ]:
df_prod

In [ ]:
df_sell

In [ ]:
df_prod_cat_name_tr

In [ ]:
liste_df = [df_customers,
    df_geoloc,
    df_order_item,
    df_order_pay,
    df_order_rew,
    df_orders,
    df_prod,
    df_sell,
    df_prod_cat_name_tr]

In [ ]:
liste_df

In [ ]:
noms_df = ['df_customers', 
           'df_geoloc',
           'df_order_items',
           'df_order_pay',
           'df_order_rew',
           'df_orders',
           'df_prod',
           'df_sell',
           'df_prod_cat_name_tr']

In [ ]:
#Dimensions jeu de données
for df in liste_df :
    print(df.shape)

In [ ]:
df_customers.shape[0]

In [ ]:
df_customers.isna().sum().sum()

In [ ]:
df.shape[0]

In [ ]:
df.shape[1]

In [ ]:
#Données manquantes
for df in liste_df:
    print(df.isna().sum().sum()/(df.shape[0]*df.shape[1]), 'NaN')

In [ ]:
#On a des NaN uniquement des order_reviews, orders et products

In [ ]:
#Doublons
for df in liste_df:
    print(df.duplicated().sum())

In [ ]:
#Fonction de nettoyage
#Contrôle


In [ ]:
order = pd.merge(df_order_item, 
        df_orders)

In [ ]:
produits_par_client = pd.merge(df_customers, order)
produits_par_client.describe(include='object')

In [ ]:
prod = df_prod[['product_id', 'product_category_name']]
prod

In [ ]:
prod_par_client = pd.merge(produits_par_client, prod)
prod_par_client = pd.merge(prod_par_client, df_prod_cat_name_tr)

In [ ]:
prod_par_client 

In [ ]:
prod_par_client.drop(columns='product_category_name', inplace=True)

In [ ]:
prod_par_client

In [ ]:
client_data = pd.merge(prod_par_client, df_order_rew)
client_data

In [ ]:
client_data = pd.merge(client_data, df_order_pay)
client_data

In [ ]:
client_data.drop(columns=['customer_zip_code_prefix'], inplace=True)

In [ ]:
client_data

In [ ]:
client_data.columns

In [ ]:
client_data['order_approved_at']

In [ ]:
client_data['order_approved_at']

In [ ]:
client_data.isna().sum()

In [ ]:
#Traitement des NaN
client_data['review_comment_title'].fillna(' ', inplace=True)
client_data['review_comment_message'].fillna(' ', inplace=True)
   
    
client_data['order_approved_at'].fillna(0, inplace=True)
client_data['order_delivered_carrier_date'].fillna(0, inplace=True)
client_data['order_delivered_customer_date'].fillna(0, inplace=True)
    

In [ ]:
client_data.isna().sum()

In [ ]:
#suppression des doublons
client_data.drop_duplicates(subset=['customer_unique_id','product_id', 'order_id'], keep='first', inplace=True)
client_data.shape

In [ ]:
client_data['product_category_name_english'].unique()

In [ ]:
#Assemblage, traitement outliers et feature engineering
#mapping categories
dict_categories = {
#home
'furniture_living_room' : 'home',
'furniture_mattress_and_upholstery' : 'home',
'furniture_bedroom' : 'home',
'furniture_decor' : 'home',
'bed_bath_table' : 'home',
'kitchen_dining_laundry_garden_furniture' : 'home',
'la_cuisine' : 'home',
'home_confort' : 'home',
'home_comfort_2' : 'home',
'christmas_supplies' : 'home',
    
#appliances
'small_appliances' : 'appliances',
'small_appliances_home_oven_and_coffee' : 'appliances',
'home_appliances_2' : 'appliances',
'home_appliances' : 'appliances',
'housewares' : 'appliances',
    
#construction
'construction_tools_construction': 'construction',
'costruction_tools_garden': 'construction',
'costruction_tools_tools': 'construction',
'construction_tools_safety': 'construction',
'construction_tools_lights': 'construction',
'home_construction': 'construction',
'air_conditioning': 'construction',

    
#office
'office_furniture' : 'office',
'industry_commerce_and_business' : 'office',
'stationery' : 'office',
'agro_industry_and_commerce' : 'office',
'signaling_and_security' : 'office',
'furnitures' : 'office',
'security_and_services' : 'office',
    
#electronics
'telephony' : 'electronics',
'electronics': 'electronics',
'computers_accessories': 'electronics',
'consoles_games': 'electronics',
'fixed_telephony': 'electronics',
'audio': 'electronics',
'computers': 'electronics',
'tablets_printing_image': 'electronics',
    
#sports_leisure
'sports_leisure' : 'sports_leisure',
'musical_instruments': 'sports_leisure',
'party_supplies': 'sports_leisure',
'luggage_accessories': 'sports_leisure',
    
#arts
'books' : 'arts',
'books_imported': 'arts',
'books_general_interest': 'arts',
'books_technical': 'arts',
'art': 'arts',
'toys': 'arts',
'cine_photo': 'arts',
'cds_dvds_musicals': 'arts',
'music': 'arts',
'dvds_blu_ray': 'arts',
'arts_and_craftmanship': 'arts',
    
#fashion
'watches_gifts' : 'fashion',
'fashion_bags_accessories' : 'fashion',
'fashion_underwear_beach' : 'fashion',
'fashion_shoes' : 'fashion', 
'fashion_male_clothing' : 'fashion',
'fashio_female_clothing' : 'fashion',
'fashion_sport' : 'fashion',
'fashion_childrens_clothes' : 'fashion',
    
#health_beauty
'health_beauty' : 'health_beauty', 
'baby': 'health_beauty',
'diapers_and_hygiene' : 'health_beauty',
'perfumery' : 'health_beauty',
    
#garden and pets
'flowers' : 'garden_pets',
'pet_shop' : 'garden_pets',
'garden_tools' : 'garden_pets',
    
#auto
'auto' : 'auto',
    
#food_drinks
'food_drink' : 'food_drinks',
'food' : 'food_drinks',
'drinks' : 'food_drinks',
    
#other
'Unknown' : 'other',
'market_place' : 'other', 
'cool_stuff' : 'other'}

In [ ]:
client_data['product_category_name_english'] = client_data['product_category_name_english'].map(dict_categories)

In [ ]:
#feature engineering

#moyen de paiement le plus utilisé
paiement = client_data.reset_index().sort_values(
        ['customer_id','payment_type','payment_installments'],ascending=False).groupby(
        ['customer_id', 'payment_type','payment_installments']).head(1)[[
        'customer_id', 'payment_type','payment_installments']]
paiement.columns = ['customer_id', 'Moy_paiement_le plus_utilisé','Facilités']
client_data = pd.merge(client_data, paiement)

#Catégorie la plus achetée
cat = client_data.sort_values(
        ['customer_id','product_category_name_english'], ascending=False).groupby(
        ['customer_id','product_category_name_english']).head(1)[[
        'customer_unique_id', 'product_category_name_english']]
cat.columns = ['customer_unique_id', 'Cat_la_plus_achetee']
client_data = pd.merge(client_data, cat)

#Mettre en bon type les dates
client_data['shipping_limit_date'] = client_data[
        'shipping_limit_date'].astype('datetime64[ns]')
client_data['order_purchase_timestamp'] = client_data[
        'order_purchase_timestamp'].astype('datetime64[ns]')
client_data['order_approved_at'] = pd.to_datetime(client_data[
        'order_approved_at'], errors='coerce')
client_data['order_delivered_carrier_date'] = pd.to_datetime(
        client_data['order_delivered_carrier_date'], errors='coerce')
client_data['order_delivered_customer_date'] = pd.to_datetime(
        client_data['order_delivered_customer_date'], errors='coerce')
client_data['order_estimated_delivery_date'] = client_data[
        'order_estimated_delivery_date'].astype('datetime64[ns]')

#Délai moyen livraison
client_data['delai_moy_livraison'] = client_data['order_delivered_customer_date'] - client_data[
        'order_purchase_timestamp']
client_data['delai_moy_livraison'] = client_data['delai_moy_livraison'].apply(lambda x: x.days)

#nombre de jours écoulés depuis dernier achat
client_data['delai_dernier_achat'] = client_data['order_purchase_timestamp'].max() -  client_data[
        'order_purchase_timestamp']
client_data['delai_dernier_achat'] = client_data['delai_dernier_achat'].apply(lambda x: x.days)

client_data

In [ ]:
client_data.isna().sum()


In [ ]:
client_data.shape

In [ ]:
#suppression valeurs aberrantes
client_data.drop(client_data[client_data['delai_moy_livraison'] < 0 ].index, inplace=True)
client_data.drop(client_data[client_data['delai_dernier_achat'] < 0 ].index, inplace=True)

In [ ]:
client_data.shape

In [ ]:
data_client = client_data[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state',
       'order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'order_status',
       'order_purchase_timestamp', 'order_approved_at',
       'order_delivered_carrier_date', 'order_delivered_customer_date',
       'order_estimated_delivery_date', 'product_category_name_english',
       'review_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value', 'Moy_paiement_le plus_utilisé',
       'Facilités', 'Cat_la_plus_achetee', 'delai_moy_livraison',
       'delai_dernier_achat']]
data_client

In [ ]:
data_client.isna().sum()

In [ ]:

#Nombre de produits achetés par client
data_client.set_index('customer_id', inplace=True)
data_client['nb_total_produits'] = data_client.groupby('customer_id')['product_id'].count()
data_client['nb_total_produits']


In [ ]:
#Montant moyen des achats
data_client['montant_moyen_achats'] = client_data.groupby(
        ['customer_id', 'order_id'])['price'].sum().groupby(['customer_id']).mean()
data_client['montant_moyen_achats']


In [ ]:
#Montant max des achats
data_client['Montant_max_achats'] = client_data.groupby(['customer_id', 'order_id'])['price'].max().groupby(['customer_id']).max()
data_client['Montant_max_achats']



In [ ]:
 #Nombre moyen de produits par commande
data_client['Nombre_moyen_prod_par_comm'] = client_data.groupby(
        ['customer_id', 'order_id'])['price'].count().groupby(['customer_id']).mean()
data_client['Nombre_moyen_prod_par_comm']


In [ ]:
#note moyenne des commentaires
data_client['note_moyenne_comm'] = client_data.groupby('customer_id')['review_score'].mean()
data_client.head()
data_client['note_moyenne_comm']

In [ ]:

data_client['montant_moyen_achats']

In [ ]:

data_client['Montant_max_achats']

In [ ]:

data_client['Nombre_moyen_prod_par_comm']

In [ ]:
data_client.isna().sum()

In [ ]:
#data_client['Cat_la_plus_achetee'] = data_client['Cat_la_plus_achetee'].agg(dict_agg1)

In [ ]:
print('###Aggrégation des features sur id client ###')
    #aggregation des clients par identifiant unique
dict_agg = {'Cat_la_plus_achetee' : lambda x: x.mode()[0],
           'customer_unique_id' : lambda x: x.mode()[0],
           'customer_city' : lambda x: x.mode()[0],
           'customer_state' : lambda x: x.mode()[0],
           'seller_id': lambda x: x.mode()[0],
           'nb_total_produits' : 'sum',
           'montant_moyen_achats' : 'mean',
           'Montant_max_achats' : 'max',
           'delai_moy_livraison' : 'mean',
           'Nombre_moyen_prod_par_comm' : 'mean',
           'delai_dernier_achat' : 'min',
           'note_moyenne_comm' : 'mean',
           'Moy_paiement_le plus_utilisé' : lambda x: x.mode()[0],
           'Facilités' : 'mean'
 }  

In [ ]:
data_client = data_client.groupby('customer_id').agg(dict_agg)

In [ ]:
data_client['nb_total_produits']

In [ ]:
data_client['montant_moyen_achats']

In [ ]:
data_client.isna().sum()

In [ ]:
data_client.shape

In [ ]:
data_client = data_client.loc[: , ['customer_unique_id', 'customer_city', 'customer_state', 'seller_id', 
                               'Cat_la_plus_achetee', 'nb_total_produits', 'montant_moyen_achats', 
                               'Montant_max_achats', 
                               'delai_moy_livraison', 'Nombre_moyen_prod_par_comm', 'delai_dernier_achat', 
                               'note_moyenne_comm', 
                               'Moy_paiement_le plus_utilisé', 'Facilités' ]]
data_client

In [ ]:
#data_client.reset_index('customer_id', inplace=True)

In [ ]:
data_client.isna().sum()

In [ ]:
data_client.shape

In [ ]:
#traitement des outliers

for column in data_client.select_dtypes(include = ['int32','float64']).columns.tolist() :
    #outliers aux extrémités des distributions
    data_client.loc[data_client[column] > data_client[column].quantile(0.99)] = np.nan
    data_client.loc[data_client[column] < data_client[column].quantile(0.01)] = np.nan
print(data_client.shape)
    

In [ ]:
data_client.isna().sum()

In [ ]:
data_client.dropna(inplace=True)

In [ ]:
data_client.shape

In [ ]:
#Dataviz
for column in data_client.select_dtypes(include = ['int32','float64']).columns.tolist() :
    plt.figure(figsize=(8,6))
    plt.title('Distribution de ' + column)
    sns.distplot(data_client[column].dropna(), bins=30)
    
for column in data_client.select_dtypes(include = ['int32','float64']).columns.tolist() :
    sns.boxplot(data_client[column])
    chaine = 'Distribution de : ' + column
    plt.title(chaine)
    plt.xlabel(column)
    plt.show()

In [ ]:
data_client.select_dtypes(['float64']).columns

In [ ]:
data_client.select_dtypes(['float64']).values

In [ ]:
%matplotlib inline
for column in data_client[['montant_moyen_achats', 'Montant_max_achats',
       'delai_moy_livraison',
       'delai_dernier_achat']]:
    f, axes = plt.subplots(1,2, figsize=(12,4))
    titre = 'Distribution de ' + str(column)
    plt.title(titre)
    sns.distplot(data_client[column], bins=30, ax=axes[0])
    titre = 'Distribution de ' + str(column)
    plt.title(titre)
    sns.boxplot(data_client[column], ax=axes[1])
    plt.show()
for column in data_client[['nb_total_produits', 'Nombre_moyen_prod_par_comm', 'note_moyenne_comm', 'Facilités']]:
    #valeurs = [0,1,2,3,4,5,6,7]
    sns.histplot(data_client[column])
    titre = 'Distribution de ' + str(column)
    plt.title(titre)
    #sns.histplot(data_client[column], bins=30, x=column)
    #titre = 'Distribution de ' + str(column)
    #plt.title(titre)
    #sns.histplot(data_client[column], x=column)
    plt.show()

In [ ]:
#corrélation probable entre montant_moyen_achats et montant_max_achats, à vérifier via l'analyse multivariée

In [ ]:
#Analyse multivariée
#Matrice de corrélation
plt.figure(figsize=(7,7))
sns.set(font_scale=1)
plt.title('Matrice de corrélation de pearson entre les différentes features')

numeric_columns = data_client.select_dtypes(include = ['int32','float64']).columns
corr = data_client[numeric_columns].corr()

mask = np.zeros_like(corr, dtype=np.bool)
mask[np.triu_indices_from(mask)] = True 

ax = sns.heatmap(corr, mask=mask, vmin=-1, cmap='coolwarm')
plt.show()
sns.set(font_scale=1)

In [ ]:
#très forte corrélation entre nombre_moyen_prod_par_comm et nb_total_produits
#forte corrélation entre montant_max_achats et montant_moyen_achats


In [ ]:
#Pour avoir un bon algorithme, on va modifier l'allure de la distribution pour obtenir une variable dont la distribution 
#se rapproche d'une loi normale
#for column in data_client[['nb_total_produits','montant_moyen_achats']] :
test1 = np.log(data_client['montant_moyen_achats'])
sns.distplot(test1)
ax = plt.gca()
ax.legend(column)


In [ ]:
test2 = np.log(data_client['Montant_max_achats'])
sns.distplot(test2)
ax = plt.gca()
ax.legend(column)

In [ ]:
test3 = np.log(1+data_client['delai_dernier_achat'])
sns.distplot(test3)
ax = plt.gca()
ax.legend(column)

In [ ]:
test4 = np.log(1+data_client['Facilités'])
sns.distplot(test4)
ax = plt.gca()
ax.legend(column)

In [ ]:
state = data_client.groupby(['customer_state']).count().sort_values(
    by='customer_city', ascending=False)['customer_city'].head(10)
plt.figure(figsize=(10,6))
plt.title('Les 10 états avec le plus de clients')
sns.barplot(x = state.values,
           y = state.index)


In [ ]:
cities = data_client.groupby(['customer_city']).count()[
    'customer_state'].sort_values(ascending=False).head(10)
plt.figure(figsize=(10,6))
plt.title('Les 10 villes avec le plus de clients')
sns.barplot(x = cities.values,
           y = cities.index)


In [ ]:
#On va conserver les variables suivantes après les transformation log
data_client['Log-montant_moyen_achats'] = test1

In [ ]:
data_client_cleaned = data_client.to_csv('data_client_cleaned.csv', index=True)
print('\nCSV String:\n', data_client_cleaned)